In [0]:
%pip install gdown


In [0]:
dbutils.library.restartPython()

In [0]:
import gdown

folder_url = "https://drive.google.com/drive/folders/1Tw7V9eBtUxy0xQMW38z3-bzWI_ewzLm6"
local_dir = "/tmp/drive_data"

files = gdown.download_folder(folder_url, output=local_dir, quiet=False, use_cookies=False)
print(files)

In [0]:
import shutil

volume_path = "/Volumes/workspace/default/usage_metrics/drive_data"

shutil.copytree(local_dir, volume_path, dirs_exist_ok=True)

display(dbutils.fs.ls(volume_path))

In [0]:
df = spark.read.csv("dbfs:/Volumes/workspace/default/usage_metrics/drive_data/custs",
                    header=False, inferSchema=True,sep=',')
df= df.toDF ("custid","fname","lname","age","profession")                   
df.printSchema()
df = df.withColumnRenamed("profession", "JobDescription")
df.where("custid in (4000002)").show()
#df.show(5)
#display(df)

In [0]:
df = spark.read.options(header=False, inferSchema=True,sep=',').csv("dbfs:/Volumes/workspace/default/usage_metrics/drive_data/")
df.printSchema()
df.show(2)
#display(df)

In [0]:
from pyspark.sql.functions import col

df = (spark.read
      .options(header=False, inferSchema=True, sep=",")
      .csv("/Volumes/workspace/default/usage_metrics/drive_data/")
      .select("*", col("_metadata.file_name").alias("source_file")))

df.groupBy("source_file").count().show(truncate=False)

In [0]:
folder = "/Volumes/workspace/default/usage_metrics/drive_data/"

files = [f for f in dbutils.fs.ls(folder) if not f.isDir()]
for f in files:
    print("=" * 60)
    print(f"File: {f.name}  ({f.size} bytes)")
    print("=" * 60)

    df = (spark.read
          .options(header=True, inferSchema=True, sep=",")
          .csv(f.path))

    df.printSchema()
    df.show(3, truncate=False)

In [0]:
path="dbfs:/Volumes/workspace/default/usage_metrics/drive_data/custs"
# 2. format + option + load
df = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(path))
df.printSchema()
df.show(2)
#df.show(5)      